In [1]:
import geopandas as gpd
import pandas as pd

In [3]:
aoi = gpd.read_file(r'Aoi\aoi.shp')

In [4]:
gdf_13 = gpd.read_file(r'RGI_glacier\nsidc0770_13.rgi60.CentralAsia\13_rgi60_CentralAsia.shp')
gdf_14 = gpd.read_file(r'RGI_glacier\nsidc0770_14.rgi60.SouthAsiaWest\14_rgi60_SouthAsiaWest.shp')
gdf_15 = gpd.read_file(r'RGI_glacier\nsidc0770_15.rgi60.SouthAsiaEast\15_rgi60_SouthAsiaEast.shp')


In [5]:
# Make sure all layers use the same CRS as the AOI
gdf_13 = gdf_13.to_crs(aoi.crs) if gdf_13.crs != aoi.crs else gdf_13
gdf_14 = gdf_14.to_crs(aoi.crs) if gdf_14.crs != aoi.crs else gdf_14
gdf_15 = gdf_15.to_crs(aoi.crs) if gdf_15.crs != aoi.crs else gdf_15

# Concatenate the three glacier layers
glaciers_all = pd.concat([gdf_13, gdf_14, gdf_15], ignore_index=True)
glaciers_all = gpd.GeoDataFrame(glaciers_all, geometry='geometry', crs=aoi.crs)

# Keep only the part within the AOI
aoi_union = aoi.union_all
glaciers_in_aoi = gpd.clip(glaciers_all, aoi)
glaciers_in_aoi

,RGIId,GLIMSId,BgnDate,EndDate,CenLon,CenLat,O1Region,O2Region,Area,Zmin,...,Aspect,Lmax,Status,Connect,Form,TermType,Surging,Linkages,Name,geometry
85646,RGI60-15.03230,G088281E27501N,20001226,-9999999,88.280970,27.501009,15,2,0.298,5048,...,234,759,0,0,0,0,9,9,NaN,"POLYGON ((88.27769 27.50258, 88.27804 27.50293..."
85647,RGI60-15.03231,G088279E27505N,20001226,-9999999,88.278790,27.505355,15,2,0.140,5144,...,208,721,0,0,0,0,9,9,NaN,"POLYGON ((88.27813 27.50694, 88.2784 27.50732,..."
85645,RGI60-15.03229,G088282E27509N,20001226,-9999999,88.281902,27.508518,15,2,0.207,5192,...,113,556,0,0,0,0,9,9,NaN,"POLYGON ((88.28074 27.50633, 88.28049 27.5068,..."
85620,RGI60-15.03204,G088274E27511N,20001226,-9999999,88.273849,27.510660,15,2,0.737,5096,...,222,1009,0,0,0,0,9,9,NaN,"POLYGON ((88.27881 27.50868, 88.27858 27.50825..."
85171,RGI60-15.02755,G088239E27510N,20001108,-9999999,88.238893,27.510350,15,2,0.168,5069,...,206,497,0,0,0,0,9,9,NaN,"POLYGON ((88.23649 27.50885, 88.23603 27.5091,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41036,RGI60-13.41039,G076414E37275N,20070729,-9999999,76.414000,37.275000,13,5,0.096,4945,...,315,356,0,0,0,0,9,9,NaN,"POLYGON ((76.41116 37.27513, 76.41117 37.27534..."
41046,RGI60-13.41049,G076483E37278N,20090921,-9999999,76.483000,37.278000,13,5,0.114,4701,...,331,404,0,0,0,0,9,9,CN5Y652E0001,"POLYGON ((76.4838 37.27969, 76.48434 37.27917,..."
41174,RGI60-13.41177,G075974E37518N,20090811,-9999999,75.974000,37.518000,13,2,0.150,4791,...,354,565,0,0,0,0,9,9,CN5Y655H0006,"POLYGON ((75.97213 37.51626, 75.97194 37.51655..."
41181,RGI60-13.41184,G076013E37530N,20090811,-9999999,76.013000,37.530000,13,2,0.243,4733,...,340,675,0,0,0,0,9,9,CN5Y655H0001,"POLYGON ((76.01234 37.52829, 76.01151 37.52844..."


In [ ]:
output_gpkg = "glaciers_in_aoi.gpkg"
glaciers_in_aoi.to_file(output_gpkg, layer="glaciers_in_aoi", driver="GPKG")
print(f"Saved: {output_gpkg}")

In [7]:
output_path = r'model_data\shapefiles\glacier\glaciers_in_aoi.gpkg'
glaciers_in_aoi.to_file(output_path, driver='GPKG', index=False)
print(f'Saved to {output_path}')

Saved to model_data\shapefiles\glacier\glaciers_in_aoi.gpkg


In [9]:
import json
import os
from shapely.geometry import shape

# Read CSV as tabular data, then parse GeoJSON geometry from '.geo'
dam_df = pd.read_csv(r'dam_characteristics_hma.csv')

dam = gpd.GeoDataFrame(
    dam_df.drop(columns=['.geo']),
    geometry=dam_df['.geo'].apply(lambda g: shape(json.loads(g))),
    crs='EPSG:4326'
)

output_dam = r'model_data\shapefiles\dam\dam_characteristics_hma.gpkg'
os.makedirs(os.path.dirname(output_dam), exist_ok=True)
dam.to_file(output_dam, layer='dam_characteristics_hma', driver='GPKG', index=False)

print(f'Saved to {output_dam}')
dam.head()

Saved to model_data\shapefiles\dam\dam_characteristics_hma.gpkg


,system:index,dam_crest_m,dam_slope_deg,dam_width_m,freeboard_m,wse_m,geometry
0,00000000000000001b02,4779.487805,0.000000,500.000000,-0.512195,4780.000000,"POLYGON ((77.17606 35.67557, 77.17625 35.67552..."
1,00000000000000001b03,3564.480000,2.050126,275.510204,-144.775649,3709.255649,"POLYGON ((73.87726 36.83662, 73.87729 36.83653..."
2,00000000000000001b04,4617.739910,0.000000,500.000000,328.739910,4289.000000,"POLYGON ((91.80518 27.82788, 91.80523 27.82781..."
3,00000000000000001b05,2719.048000,0.000000,500.000000,162.048000,2557.000000,"POLYGON ((74.87688 36.45683, 74.87691 36.45678..."
4,00000000000000001b06,4808.394495,0.000000,500.000000,201.394495,4607.000000,"POLYGON ((86.06279 28.07843, 86.0628 28.07842,..."
